In [ ]:
import os
from dotenv import load_dotenv

from pathlib import Path
import zipfile
import json
import re
from collections import defaultdict

from datasets import Dataset

load_dotenv()
DATA_ROOT = Path(os.environ["DATA_ROOT"])
os.environ["HF_HOME"] = Path(DATA_ROOT, ".hf_cache")

# 데이터 확인

In [4]:
sample_zip_path = Path(DATA_ROOT, "data", "Training", "Orig", "TS_TS1_EG_원자력_EG10_핵융합.zip")
label_zip_path = Path(DATA_ROOT, "data", "Training", "Label", "TL_TL1_EG_원자력_EG10_핵융합.zip")

def check_zip_file(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        print(f"총 {len(names)}개 파일")
        for name in names[:10]:
            print(name)

check_zip_file(sample_zip_path)
print("-"*30)
check_zip_file(label_zip_path)

총 1108개 파일
/jp1986157746b2.json
/jp1985099312b2.json
/jp1985054526b2.json
/jp1986149675b2.json
/jp1989315282b2.json
/jp1987242795b2.json
/jp1985167590b2.json
/jp1985155953b2.json
/jp1986041696b2.json
/jp1985191577b2.json
------------------------------
총 1108개 파일
/jp1984019373b2.json
/jp1989102401b2.json
/jp1985099312b2.json
/jp1987172975b2.json
/jp1986186780b2.json
/jp1985155953b2.json
/jp1985054526b2.json
/jp1986242250b2.json
/jp1985224756b2.json
/jp1986154778b2.json


In [5]:
with zipfile.ZipFile(sample_zip_path, "r") as zf:
    with zf.open("/jp1986157746b2.json") as f:
        org_data = json.load(f)

print(org_data["dataset"])

{'application_year': '1986', 'invention_title': '제어형 핵융합 원자로의 플라즈마 피복벽 및 상기 피복벽의 제법', 'ipc_section': 'G', 'register_date': '19950607', 'ipc_subclass': 'G21B', 'applicant_name': '스탠·안데유스토리이', 'ipc_main': 'G21B-001/00', 'register_year': '1995', 'ipc_all': 'G21B-001/00', 'ipc_class': 'G21', 'claims': '냉각수를 도입하는 도입 매니폴드와, 상기 냉각수를 배출하는 배출 매니폴드와, 상기 도입 매니폴드 및 상기 배출 매니폴드와 협동해 가압수에 의한 냉각 회로를 구성하는 원형 단면의 길이 방향 관통 덕트를 갖고 있고, 한 개의 측면이 플라즈마로 면 해야 할 침식층을 형성하는 봉상 부재의 복수와, 상기 막대모양부재의 각각에 있어서 상기 한 개의 측면에 인접하는 두 개의 측면의 각각으로부터 돌출되도록 일체적에 설치된 플랜지를 구비하고 있으며, 상기 막대모양부재는 플라즈마 수용실을 확정하기 위해, 상기 플랜지가 서로 마주 보도록 연속적으로 배치됨과 동시에 인접하는 플랜지가 서로 용접되어 플라즈마를 밀폐하는 밀폐 방호층을 형성해 있고, 상기 막대모양부재 및 상기 플랜지의 각각은, 환바닥부를 가지는 긴방향 홈을 인접하는 봉상 부재 및 플랜지와 협동해 상기 침식층 측에 형성 하는, 제어형 핵융합 원자로의 플라즈마 피복벽. 가압수에 의한 냉각 회로를 구성하는 원형 단면의 길이 방향 관통 덕트, 플라즈마에 접하도록 한 개의 측면에 형성된 침식층, 및 상기 한 개의 측면에 인접하는 두 개의 측면의 각각으로 돌출되는 플랜지를 갖고 있고, 환바닥부를 가지는 긴방향 홈을 인접하는 봉상 부재와 협동해 상기 침식층 측에 형성 하는 봉상 부재를 인발나무 및 압연 가공에 의해서 복수 형성 할 단계와, 플라즈마 수용실을 확정하기 위해 상기 막대모양부재를 상기 플랜지가 

In [7]:
with zipfile.ZipFile(label_zip_path, "r") as zf:
    with zf.open("/jp1986157746b2.json") as f:
        label_data = json.load(f)

print(label_data["dataset"])

{'country_code': 'KR', 'updateDate': '20231207', 'Mtext': '핵융합', 'documentId': 'jp1986157746b2', 'application_number': '1986-157746', 'Lno': 'EG', 'Ltext': '원자력', 'keyword': 'ET/에너지·자원', 'ipc_main': 'G21B-001/00', 'document_type': '2', 'Mno': 'EG10'}


In [15]:
"_".join([
    str(label_data["dataset"]["Mno"]),
    str(label_data["dataset"]["Mtext"])
])

'EG10_핵융합'

라벨 - 중분류

# 정제 텍스트 데이터셋

#### 라벨 추출

In [ ]:
DATA = Path(DATA_ROOT, "data")
MNO_RE = re.compile(r"[A-Z]{2}[0-9]{2}")          # EG10 형태

def build_label_index():
    doc2mno = defaultdict(set)           # documentId -> {Mno}
    for split in ["Training", "Validation"]:       # 폴더 구분 무시하고 전부 모음
        for z in (DATA / split / "Label").glob("*.zip"):
            mno = MNO_RE.findall(z.stem)[0]         # zip명에서 Mno
            with zipfile.ZipFile(z) as zf:
                for file_name in zf.namelist():
                    if file_name.endswith(".json"):
                        doc2mno[Path(file_name).stem].add(mno)
    return doc2mno

doc2mno = build_label_index()
doc2mno

In [26]:
MNO2ID = {mno: id_num for id_num, mno in enumerate(sorted({mno for mno_set in doc2mno.values() for mno in mno_set}))}  # 188개
ID2MNO = {id_num: mno for mno, id_num in MNO2ID.items()}
MNO2LNO = {mno: mno[:2] for mno in MNO2ID}

In [28]:
MNO2ID

{'EA01': 0,
 'EA02': 1,
 'EA03': 2,
 'EA04': 3,
 'EA05': 4,
 'EA06': 5,
 'EA07': 6,
 'EA08': 7,
 'EA09': 8,
 'EA10': 9,
 'EA11': 10,
 'EA12': 11,
 'EA13': 12,
 'EA14': 13,
 'EA15': 14,
 'EB01': 15,
 'EB02': 16,
 'EB03': 17,
 'EB04': 18,
 'EB05': 19,
 'EB06': 20,
 'EB07': 21,
 'EB08': 22,
 'EC01': 23,
 'EC02': 24,
 'EC03': 25,
 'EC04': 26,
 'EC05': 27,
 'EC06': 28,
 'EC07': 29,
 'EC08': 30,
 'EC09': 31,
 'EC10': 32,
 'EC11': 33,
 'ED01': 34,
 'ED02': 35,
 'ED03': 36,
 'ED04': 37,
 'ED05': 38,
 'ED06': 39,
 'ED07': 40,
 'ED08': 41,
 'ED09': 42,
 'ED10': 43,
 'ED11': 44,
 'EE01': 45,
 'EE02': 46,
 'EE03': 47,
 'EE04': 48,
 'EE05': 49,
 'EE06': 50,
 'EE07': 51,
 'EE08': 52,
 'EE09': 53,
 'EE10': 54,
 'EE11': 55,
 'EE12': 56,
 'EE13': 57,
 'EE14': 58,
 'EF01': 59,
 'EF02': 60,
 'EF03': 61,
 'EF04': 62,
 'EF05': 63,
 'EF06': 64,
 'EF07': 65,
 'EG01': 66,
 'EG02': 67,
 'EG03': 68,
 'EG04': 69,
 'EG05': 70,
 'EG06': 71,
 'EG07': 72,
 'EG08': 73,
 'EG09': 74,
 'EG10': 75,
 'EH01': 76,
 'EH02': 

#### 데이터셋

In [ ]:
TEXT_FIELDS = ["invention_title", "abstract", "claims", "ipc_main"]
# INFO_FIELDS = ["documentId", "application_number", "application_year", "application_date", "register_number", "register_date", "resgister_year", "ipc_section", "ipc_subclass", "ipc_all", "ipc_class"]

def iter_clean_records(doc2mno):
    seen = set()
    for split in ["Training", "Validation"]:
        for z in (DATA / split / "Orig").glob("*.zip"):
            with zipfile.ZipFile(z) as zf:
                for file_name in zf.namelist():
                    if not file_name.endswith(".json"):
                        continue
                    doc_id = Path(file_name).stem
                    if doc_id in seen:                # 여러 zip 중복 수록 → 1회만
                        continue
                    seen.add(doc_id)
                    data = json.loads(zf.read(file_name))["dataset"]
                    mnos = sorted(doc2mno[doc_id])
                    yield {
                        "document_id": doc_id,
                        **{field: data.get(field) or "" for field in TEXT_FIELDS},   # 결측 → 빈 문자열
                        "mno": mnos,                                  # ["EG10", ...]
                        "label_ids": [MNO2ID[m] for m in mnos],       # [75, ...]
                        "lno": sorted({MNO2LNO[m] for m in mnos}),
                    }

In [ ]:
type(iter_clean_records(doc2mno))

generator

In [33]:
ds = Dataset.from_generator(lambda: iter_clean_records(doc2mno)) # 문서당 1행
tmp = ds.train_test_split(test_size=0.1, seed=42)
val_test = tmp["test"].train_test_split(test_size=0.5, seed=42)
splits = {"train": tmp["train"], "validation": val_test["train"], "test": val_test["test"]}
splits

Generating train split: 0 examples [00:00, ? examples/s]

{'train': Dataset({
     features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
     num_rows: 201895
 }),
 'validation': Dataset({
     features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
     num_rows: 11216
 }),
 'test': Dataset({
     features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno'],
     num_rows: 11217
 })}

In [34]:
for name, part in splits.items():
    part.push_to_hub("ingyoun/patent-clean-text", split=name)   # 미토큰화 parquet

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

...

Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            